In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
iris = load_iris()

df1 = pd.DataFrame(iris.data,columns=iris.feature_names)
df2 = pd.DataFrame(iris.target,columns=["Species"])

In [3]:
df = pd.concat([df1,df2],axis=1)

In [4]:
df = df.head(60)
df.tail(15)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),Species
45,4.8,3.0,1.4,0.3,0
46,5.1,3.8,1.6,0.2,0
47,4.6,3.2,1.4,0.2,0
48,5.3,3.7,1.5,0.2,0
49,5.0,3.3,1.4,0.2,0
50,7.0,3.2,4.7,1.4,1
51,6.4,3.2,4.5,1.5,1
52,6.9,3.1,4.9,1.5,1
53,5.5,2.3,4.0,1.3,1
54,6.5,2.8,4.6,1.5,1


In [5]:
df.Species.value_counts()

Species
0    50
1    10
Name: count, dtype: int64

## Resampling

pip install imblearn

In [6]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

In [7]:
X = df.drop(columns=["Species"])
y = df.Species

In [8]:
# Random Oversampling
oversampler = RandomOverSampler(random_state=42)
X_over, y_over = oversampler.fit_resample(X, y)

# Display class distribution after oversampling
print("Class distribution after oversampling:", Counter(y_over))

Class distribution after oversampling: Counter({0: 50, 1: 50})


In [9]:
# Random Undersampling
undersampler = RandomUnderSampler(random_state=42)
X_under, y_under = undersampler.fit_resample(X, y)

# Display class distribution after undersampling
print("Class distribution after undersampling:", Counter(y_under))

Class distribution after undersampling: Counter({0: 10, 1: 10})


## SMOTE

In [10]:
from imblearn.over_sampling import SMOTE

# Resampling the minority class. 
sm = SMOTE(sampling_strategy='minority', random_state=42)

In [11]:
# Fit the model to generate the data.
oversampled_X, oversampled_Y = sm.fit_resample(X,y)

oversampled = pd.concat([pd.DataFrame(oversampled_Y), pd.DataFrame(oversampled_X)], axis=1)

In [12]:
oversampled

,Species,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,0,5.100000,3.500000,1.400000,0.200000
1,0,4.900000,3.000000,1.400000,0.200000
2,0,4.700000,3.200000,1.300000,0.200000
3,0,4.600000,3.100000,1.500000,0.200000
4,0,5.000000,3.600000,1.400000,0.200000
...,...,...,...,...,...
95,1,5.270462,2.714092,3.984555,1.385908
96,1,6.920879,3.140659,4.680220,1.380220
97,1,6.955270,3.200000,4.685090,1.407455
98,1,5.492132,2.301311,3.990821,1.296066


In [13]:
oversampled.Species.value_counts()

Species
0    50
1    50
Name: count, dtype: int64

## BalancedBaggingClassifier

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [24]:
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.tree import DecisionTreeClassifier

#Create an instance
classifier = BalancedBaggingClassifier(base_estimator=DecisionTreeClassifier(),
                                sampling_strategy='not majority',
                                replacement=False,
                                random_state=42)
classifier.fit(X_train, y_train)

preds = classifier.predict(X_test)

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/imblearn/ensemble/_bagging.py:362: FutureWarning: `base_estimator` was renamed to `estimator` in version 0.10 and will be removed in 0.12.
  warnings.warn(


In [25]:
print("Model prediction:",preds)
print("Original values :",y_test)

Model prediction: [0 0 1 0 0 0 0 0 0 0 0 0]
Original values : 21    0
33    0
57    1
27    0
49    0
13    0
34    0
39    0
32    0
26    0
31    0
40    0
Name: Species, dtype: int64


In [26]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,preds)

1.0

In [27]:
confusion_matrix(preds,y_test)

array([[11,  0],
       [ 0,  1]])

In [28]:
print(classification_report(preds,y_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
           1       1.00      1.00      1.00         1

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

